# Lahore AQI - exploratory analysis

Four years of hourly data from the CAMS reanalysis, converted to US EPA AQI.

The point of this notebook is to answer three questions before any modelling happens:

1. What does the AQI actually look like here - level, spread, how bad the tail is
2. Which cycles are real and strong enough to be worth encoding as features
3. How far ahead is the signal still there at all, which sets a ceiling on what a
   3-day forecast can possibly achieve

Run the backfill first:

```
AQI_OFFLINE=1 python -m aqi.pipelines.backfill --start 2022-08-01
```

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from aqi.aqi_math import CATEGORY_COLOURS, category
from aqi.dataset import load_history

plt.rcParams.update({"figure.figsize": (12, 4), "axes.grid": True, "grid.alpha": 0.25})

df = load_history()
df["local"] = df["ts"].dt.tz_localize("UTC").dt.tz_convert("Asia/Karachi")
print(f"{len(df):,} rows, {df.ts.min()} to {df.ts.max()}")
df[["ts", "aqi", "pm25", "pm10", "temp", "wind_speed"]].tail()

## 1. The distribution

First thing to check: is this plausible? Lahore is consistently in the world's ten
most polluted cities, so a median in the Unhealthy band is expected. If the median
came out Good, something upstream is broken.

In [ ]:
print(df["aqi"].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).round(1))

share = df["aqi"].map(category).value_counts(normalize=True).mul(100).round(1)
print("\nHours in each category (%):")
print(share.to_string())

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(df["aqi"].dropna(), bins=70, color="#37474f")
for edge in (50, 100, 150, 200, 300):
    ax[0].axvline(edge, color="#d64541", lw=0.8, ls="--")
ax[0].set_title("AQI distribution, with EPA category edges")
ax[0].set_xlabel("AQI")

share.reindex([c for c in CATEGORY_COLOURS if c in share.index]).plot.barh(
    ax=ax[1], color=[CATEGORY_COLOURS[c] for c in share.index if c in CATEGORY_COLOURS], edgecolor="#333"
)
ax[1].set_title("Share of hours by category")
ax[1].set_xlabel("% of hours")
plt.tight_layout()

## 2. Seasonality

The thing everyone in Lahore already knows: winter is far worse than summer. Crop
residue burning across Punjab starts in October, and the winter temperature
inversion then traps everything near ground level.

Worth quantifying, because if the seasonal swing is large relative to the day-to-day
variation then day-of-year is doing most of the forecasting work and the model needs
to be judged against a seasonal baseline, not just persistence.

In [ ]:
monthly = df.groupby(df["local"].dt.month)["aqi"].agg(["median", "mean", "std", "max"]).round(1)
monthly.index.name = "month"
print(monthly.to_string())

swing = monthly["median"].max() - monthly["median"].min()
daily_sd = df.groupby(df["local"].dt.date)["aqi"].mean().std()
print(f"\nSeasonal swing (max - min monthly median): {swing:.0f} AQI")
print(f"Std dev of daily means:                    {daily_sd:.0f} AQI")
print("Seasonality is", "larger" if swing > daily_sd else "smaller", "than day-to-day variation")

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(13, 7), sharex=False)

daily = df.set_index("local")["aqi"].resample("D").mean()
ax[0].plot(daily.index, daily.values, lw=0.7, color="#37474f")
ax[0].plot(daily.index, daily.rolling(30, center=True).mean(), lw=2.2, color="#d64541", label="30-day mean")
ax[0].axhline(200, color="#8f3f97", ls="--", lw=1, label="Unhealthy")
ax[0].set_title("Daily mean AQI")
ax[0].legend()

by_hour = df.groupby(df["local"].dt.hour)["aqi"]
hours = by_hour.median()
ax[1].plot(hours.index, hours.values, marker="o", color="#37474f")
ax[1].fill_between(hours.index, by_hour.quantile(0.25), by_hour.quantile(0.75), alpha=0.2, color="#37474f")
ax[1].set_title("Diurnal cycle (local time) - median with interquartile band")
ax[1].set_xlabel("hour of day")
ax[1].set_xticks(range(0, 24, 2))
plt.tight_layout()

The diurnal shape is the giveaway for a traffic-and-inversion driven city: a morning
peak as rush hour coincides with a shallow overnight boundary layer, a midday dip as
the layer deepens and disperses everything, then a slow climb through the evening.

This is why `hour_sin`/`hour_cos` are in the feature set rather than a raw integer hour.

## 3. Autocorrelation - how far ahead is there any signal?

This is the single most important plot in the notebook. It sets the ceiling.

If AQI at t+72h is still strongly correlated with AQI at t, a 3-day forecast is a
reasonable ask. If the correlation has collapsed to near zero by then, no amount of
model capacity will fix it and the honest answer is that day 3 is barely better than
quoting the seasonal average.

In [ ]:
series = df.set_index("ts")["aqi"].asfreq("h")
lags = list(range(1, 24)) + list(range(24, 24 * 8, 6))
acf = [series.autocorr(lag) for lag in lags]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(lags, acf, marker=".", color="#37474f")
for d, c in zip((24, 48, 72), ("#2d8a5f", "#e08b00", "#d64541")):
    ax.axvline(d, color=c, ls="--", lw=1.2)
    ax.annotate(f"+{d // 24}d: r={series.autocorr(d):.2f}", (d, 0.92), color=c, fontsize=9, ha="left")
ax.axhline(0, color="#999", lw=0.8)
ax.set(title="Autocorrelation of hourly AQI", xlabel="lag (hours)", ylabel="correlation", ylim=(-0.2, 1.0))
plt.tight_layout()

print("Correlation of the *daily mean* target with the current reading:")
for h in (1, 2, 3):
    target = series.rolling(24).mean().shift(-24 * h)
    print(f"  day +{h}: r = {series.corr(target):.3f}")

## 4. What drives it

Correlations against the day+1 target. These are linear and pairwise, so they will
understate anything that acts through an interaction - but they are enough to tell
us whether the weather variables are worth carrying at all.

In [ ]:
candidates = [
    "aqi", "aqi_mean_24h", "aqi_mean_72h", "aqi_delta_24h", "aqi_std_24h",
    "temp", "humidity", "pressure", "wind_speed", "precip", "blh",
    "wind_mean_24h", "precip_sum_24h", "hours_since_rain",
    "f1_wind_mean", "f1_precip_sum", "f1_blh_min", "f1_temp_mean",
]
available = [c for c in candidates if c in df.columns]

corr = df[available + ["y_d1"]].corr()["y_d1"].drop("y_d1").sort_values()

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(corr.index, corr.values, color=["#2d8a5f" if v < 0 else "#d64541" for v in corr.values])
ax.axvline(0, color="#333", lw=0.8)
ax.set(title="Correlation with tomorrow's mean AQI", xlabel="Pearson r")
plt.tight_layout()

print(corr.round(3).to_string())

Boundary layer height and wind speed should both come out negative - more mixing
volume and more ventilation, less concentration. Temperature is negative largely
because it is a proxy for season here, not because heat cleans the air.

## 5. Does rain actually help?

Wet deposition should scrub particulates out of the air. Worth checking directly,
because if the effect is real and large it justifies `hours_since_rain` and the
forward precipitation features.

In [ ]:
rainy = df["precip_sum_24h"] > 1.0
print(f"Wet days (>1mm in 24h): {rainy.mean() * 100:.1f}% of hours")
print(f"  mean AQI when wet: {df.loc[rainy, 'aqi'].mean():.0f}")
print(f"  mean AQI when dry: {df.loc[~rainy, 'aqi'].mean():.0f}")

# Same comparison inside the smog season only, so the answer is not just
# "it rains in the monsoon and the monsoon has cleaner air anyway".
smog = df["is_smog_season"] == 1
print("\nSmog season only (Oct-Feb):")
print(f"  wet: {df.loc[smog & rainy, 'aqi'].mean():.0f}   dry: {df.loc[smog & ~rainy, 'aqi'].mean():.0f}")

bins = pd.cut(df["hours_since_rain"], [0, 6, 12, 24, 48, 96, 240, 10_000])
print("\nMean AQI by hours since last rain:")
print(df.groupby(bins, observed=True)["aqi"].agg(["mean", "count"]).round(0).to_string())

## 6. The hazardous tail

The alerting feature only matters for the top of the distribution, so it is worth
knowing how often we are in it and whether those episodes cluster (predictable) or
arrive as isolated spikes (much harder).

In [ ]:
daily = df.set_index("local")["aqi"].resample("D").mean().dropna()
bad = daily >= 200

runs = (bad != bad.shift()).cumsum()[bad]
lengths = runs.groupby(runs).size()

print(f"Days at Unhealthy or worse: {bad.sum()} of {len(daily)} ({bad.mean() * 100:.1f}%)")
print(f"Distinct episodes:          {len(lengths)}")
print(f"Median episode length:      {lengths.median():.0f} days")
print(f"Longest episode:            {lengths.max():.0f} days")
print("\nEpisode length distribution:")
print(lengths.value_counts().sort_index().to_string())

Multi-day episodes are good news for the forecast: bad air arrives as a regime that
persists, not as one-off spikes. That is precisely the structure a 3-day model can
capture, and it is why the alerting rule (which needs the lower bound of the
prediction interval to clear the threshold) is workable rather than a coin flip.

---

## What this means for the modelling

- Median sits in the Unhealthy band, so a model that only ever predicts "bad" is
  already right a lot of the time. **Persistence has to be the baseline**, and skill
  is measured against it rather than against zero.
- Strong daily and annual cycles justify the cyclical encodings and the smog-season flag.
- Autocorrelation is still meaningful at 72h, so a 3-day forecast is a fair ask -
  but the drop-off across horizons is the reason each horizon gets its own model
  rather than one model with a horizon feature.
- Wind, mixing height and rain all matter, which is what motivates carrying the
  forward weather forecast into the feature set - and the ablation in the training
  pipeline is what keeps that honest.